In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import requests

API_KEY="KaoPWHRgpK3iW9jEvnkkCCEK3w4pd3NQwNJj5mxs"
url="https://api.eia.gov/v2/electricity/rto/region-data"

response = requests.get(
    url,
    params={"api_key":API_KEY}
)
response.raise_for_status()

data=response.json()

data

StatementMeta(, 5af4d965-b72e-4b7e-a074-97945718eb09, 3, Finished, Available, Finished, False)

{'response': {'id': 'region-data',
  'name': 'Hourly Demand, Demand Forecast, Generation, and Interchange',
  'description': 'Hourly demand, day-ahead demand forecast, net generation, and interchange by balancing authority.  \n    Source: Form EIA-930\n    Product: Hourly Electric Grid Monitor',
  'frequency': [{'id': 'hourly',
    'alias': 'hourly (UTC)',
    'description': 'One data point for each hour in UTC time.',
    'query': 'H',
    'format': 'YYYY-MM-DD"T"HH24'},
   {'id': 'local-hourly',
    'alias': 'hourly (Local Time Zone)',
    'description': 'One data point for each hour in local time.',
    'query': 'LH',
    'format': 'YYYY-MM-DD"T"HH24TZH'}],
  'facets': [{'id': 'respondent',
    'description': 'Balancing Authority / Region'},
   {'id': 'type', 'description': 'Metric'}],
  'data': {'value': {'aggregation-method': 'SUM',
    'alias': 'Demand, Demand Forecast, Net Generation, or Interchange',
    'units': 'megawatthours'}},
  'startPeriod': '2019-01-01T00',
  'endPeriod

In [10]:
import requests
import json

API_KEY = "KaoPWHRgpK3iW9jEvnkkCCEK3w4pd3NQwNJj5mxs"

url = "https://api.eia.gov/v2/electricity/rto/region-data/data/"

start_date = "2026-08-01T00"
end_date = "2026-09-01T00"

all_rows = []
offset = 0
page_size = 5000

while True:

    params = {
        "api_key": API_KEY,
        "frequency": "hourly",
        "data[]": "value",
        "start": start_date,
        "end": end_date,
        "length": page_size,
        "offset": offset
    }

    response = requests.get(url, params=params)

    print("Status:", response.status_code)

    response.raise_for_status()

    result = response.json()

    rows = result["response"]["data"]

    print(f"Offset {offset}: {len(rows)} rows")

    all_rows.extend(rows)

    if len(rows) < page_size:
        break

    offset += page_size

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 12, Finished, Available, Finished, False)

Status: 200
Offset 0: 5000 rows
Status: 200
Offset 5000: 5000 rows
Status: 200
Offset 10000: 5000 rows
Status: 200
Offset 15000: 5000 rows
Status: 200
Offset 20000: 5000 rows
Status: 200
Offset 25000: 5000 rows
Status: 200
Offset 30000: 5000 rows
Status: 200
Offset 35000: 5000 rows
Status: 200
Offset 40000: 5000 rows
Status: 200
Offset 45000: 5000 rows
Status: 200
Offset 50000: 5000 rows
Status: 200
Offset 55000: 5000 rows
Status: 200
Offset 60000: 5000 rows
Status: 200
Offset 65000: 5000 rows
Status: 200
Offset 70000: 5000 rows
Status: 200
Offset 75000: 5000 rows
Status: 200
Offset 80000: 5000 rows
Status: 200
Offset 85000: 5000 rows
Status: 200
Offset 90000: 5000 rows
Status: 200
Offset 95000: 5000 rows
Status: 200
Offset 100000: 5000 rows
Status: 200
Offset 105000: 5000 rows
Status: 200
Offset 110000: 5000 rows
Status: 200
Offset 115000: 5000 rows
Status: 200
Offset 120000: 5000 rows
Status: 200
Offset 125000: 5000 rows
Status: 200
Offset 130000: 5000 rows
Status: 200
Offset 135000:

In [12]:
df = spark.createDataFrame(all_rows)

display(df.limit(20))

print("Rows:", df.count())

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f61938dc-52c6-4b9a-b801-f2440a36a46d)

Rows: 207492


In [14]:
from pyspark.sql import functions as F 
bronz_df=(df.withColumn("ingestion-timestamp", F.current_timestamp())
.withColumn("Source",F.lit("EIA API V2")))
display(bronz_df.limit(10))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fd1575bb-7406-47c2-a5da-43825d9535fe)

In [15]:
bronz_df.write.format("delta").mode("append").saveAsTable("Bronze.bronze_electricity_region_data")

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 17, Finished, Available, Finished, False)

**Silver layer**

In [16]:
bronz_df=spark.table("Bronze.bronze_electricity_region_data")
display(bronz_df.limit(50))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7e004217-dc71-4c5a-b114-06c14cf1b18a)

In [17]:
from pyspark.sql import functions as F
silver_df=(
    bronz_df.withColumn("period", F.to_timestamp("period","yyyy-MM-dd'T'HH"))
    .withColumn("value",F.col("value").cast("double"))
    .withColumn("respondent",F.upper(F.trim("respondent")))
    .withColumn("type", F.upper(F.trim("type")))
    )

display(silver_df.limit(10))  

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d4b07285-4776-46ee-9567-3f4e64293c6c)

In [7]:
silver_df.printSchema()

StatementMeta(, 0943a0ed-c826-4f31-a84b-4dbfb9ad4b67, 9, Finished, Available, Finished, False)

root
 |-- period: timestamp (nullable = true)
 |-- respondent: string (nullable = true)
 |-- respondent-name: string (nullable = true)
 |-- type: string (nullable = true)
 |-- type-name: string (nullable = true)
 |-- value: double (nullable = true)
 |-- value-units: string (nullable = true)
 |-- ingestion-timestamp: timestamp (nullable = true)
 |-- Source: string (nullable = true)



In [8]:
silver_df.filter(F.col("value")<0).show()

StatementMeta(, 0943a0ed-c826-4f31-a84b-4dbfb9ad4b67, 10, Finished, Available, Finished, False)

+-------------------+----------+--------------------+----+-----------------+------+-------------+--------------------+----------+
|             period|respondent|     respondent-name|type|        type-name| value|  value-units| ingestion-timestamp|    Source|
+-------------------+----------+--------------------+----+-----------------+------+-------------+--------------------+----------+
|2026-09-01 05:00:00|      AECI|Associated Electr...|  TI|Total interchange|-513.0|megawatthours|2026-09-13 21:19:...|EIA API V2|
|2026-09-01 05:00:00|       AVA|  Avista Corporation|  TI|Total interchange|-586.0|megawatthours|2026-09-13 21:19:...|EIA API V2|
+-------------------+----------+--------------------+----+-----------------+------+-------------+--------------------+----------+



In [9]:
silver_df.groupBy("period","respondent","type").count().filter(F.col("count")>1).show()

StatementMeta(, 0943a0ed-c826-4f31-a84b-4dbfb9ad4b67, 11, Finished, Available, Finished, False)

+------+----------+----+-----+
|period|respondent|type|count|
+------+----------+----+-----+
+------+----------+----+-----+



In [18]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("Silver.silver_electricity_region_data")

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 20, Finished, Available, Finished, False)

In [21]:
silver_df=spark.table("Silver.silver_electricity_region_data")
display(silver_df.limit(20))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ab80afd5-f564-4e0c-8efa-b417b1c470c9)

In [13]:
from pyspark.sql import functions as F
nullv_check=silver_df.select([
    F.count(F.when(F.col(c).isNull(),c)).alias(c)
    for c in silver_df.columns
])

display(nullv_check)

StatementMeta(, b2be3949-b1fa-4ec9-93d0-4a6140f46801, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cdbf3e88-05de-4f76-b481-4aad043ca4e0)

In [15]:
negative_check=silver_df.filter((F.col("type").isin("D","DF","NG")) & (F.col("value")<0))
display(negative_check)

StatementMeta(, b2be3949-b1fa-4ec9-93d0-4a6140f46801, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ed84a1af-5728-41dd-ad2d-502f740c367e)

In [16]:
negative_ng=silver_df.filter((F.col("type")=="NG")& (F.col("value")<0))
display(negative_check)

StatementMeta(, b2be3949-b1fa-4ec9-93d0-4a6140f46801, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 40469783-fbcc-4198-9a5d-aad5caab08c5)

In [17]:
duplicates=(silver_df.groupBy("period","respondent","type").count()
.filter(F.col("count")>1))
display(duplicates)

StatementMeta(, b2be3949-b1fa-4ec9-93d0-4a6140f46801, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, be3e02d9-70dd-4572-836f-2bbe44c51028)

In [2]:
silver_df.printSchema()

StatementMeta(, 12c62aeb-ce8e-4cc6-9c49-d0b34aaab017, 4, Finished, Available, Finished, False)

root
 |-- period: timestamp (nullable = true)
 |-- respondent: string (nullable = true)
 |-- respondent-name: string (nullable = true)
 |-- type: string (nullable = true)
 |-- type-name: string (nullable = true)
 |-- value: double (nullable = true)
 |-- value-units: string (nullable = true)
 |-- ingestion-timestamp: timestamp (nullable = true)
 |-- Source: string (nullable = true)



In [22]:
from pyspark.sql import functions as F

gold_fact_hourly=silver_df.groupBy("period","respondent").pivot("type").agg(F.first("value")).withColumnsRenamed({'D':'Demand','DF':'Forecast','NG':'Generation','TI':'Interchange'})
display(gold_fact_hourly.limit(20))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e6de2ce0-7ee1-46ea-b8a4-a6ad2a8434b1)

In [23]:
spark.sql("CREATE SCHEMA IF NOT EXISTS Gold")
gold_fact_hourly.write.format("delta").mode("overwrite").saveAsTable("Gold.gold_fact_hourly")

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 25, Finished, Available, Finished, False)

In [25]:
silver_df=spark.table("silver.silver_electricity_region_data")
display(silver_df.limit(10))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d92e51ba-29c9-46d0-b920-308905f1bff9)

In [26]:
dim_region=(silver_df.select("respondent","respondent-name").dropDuplicates(["respondent"]))
display(dim_region.limit(20))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0a13aec4-07c2-44bd-a80e-7fda17a7f19d)

In [27]:
dim_region.write.format("delta").mode("overwrite").saveAsTable("Gold.dim_region")

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 29, Finished, Available, Finished, False)

In [29]:
from pyspark.sql import functions as F
fact_gold_hourly=spark.table("Gold.gold_fact_hourly")
display(fact_gold_hourly.limit(10))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bbceef11-e588-4cbd-b8f9-ba2d23118856)

In [30]:
dim_date=(fact_gold_hourly.select(F.to_date("period").alias("date")).distinct()
.withColumn("year",F.year("date"))
.withColumn("month",F.month("date"))
.withColumn("month_name",F.date_format("date","MMMM"))
.withColumn("day",F.dayofmonth("date"))
.withColumn("day-of-week",F.date_format("date","EEEE")))

display(dim_date.limit(20))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 07fbe75c-08bd-4dd1-895d-d8c58d2063d9)

In [31]:
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("Gold.dim_date")

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 33, Finished, Available, Finished, False)

In [32]:
display(dim_date.limit(20))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 400e662c-66d6-4dd4-a780-6d248c32ee2b)

In [34]:
from pyspark.sql import functions as F
gold_fact_hour=spark.table('Gold.gold_fact_hourly')

gold_fact_hour=(gold_fact_hour.withColumn("date",F.to_date("period")))

display(gold_fact_hour.limit(10))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4fcdf581-88cf-48b3-8da7-5da15c08b81e)

In [36]:
gold_fact_hour.write.format("delta").mode("overwrite").option("overwriteschema","true").saveAsTable("Gold.gold_fact_hourly")

df_f=spark.table('Gold.gold_fact_hourly')
display(df_f.limit(10))


StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 38, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 73b53fe2-f3af-4dca-869d-16e7f8dfceb2)

In [37]:

df_f=spark.table('Gold.gold_fact_hourly')
display(df_f.limit(10))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 39, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 23f6e20a-369f-4092-b590-bc8428937d76)

In [41]:
fact_df = spark.table("Gold.gold_fact_hourly")
region_df = spark.table("Gold.dim_region")
date_df = spark.table("Gold.dim_date")

print("Fact rows:", fact_df.count())
print("Regions:", region_df.count())
print("Dates:", date_df.count())

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 43, Finished, Available, Finished, False)

Fact rows: 55173
Regions: 75
Dates: 32


In [42]:
print("fact__gold_hourly rows:", fact__gold_hourly.count())

print("dim_date dataframe rows:", dim_date.count())

print(
    "saved Gold.dim_date rows:",
    spark.table("Gold.dim_date").count()
)

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 44, Finished, Available, Finished, False)

fact__gold_hourly rows: 55170
dim_date dataframe rows: 32
saved Gold.dim_date rows: 32


In [43]:
duplicates=(gold_fact_hour.groupBy("period","respondent").count().filter(F.col("count")>1))
display("dub",duplicates.count())

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 45, Finished, Available, Finished, False)

'dub'

In [20]:
missing_respondent=(gold_fact_hour.select("respondent").distinct().join(region_df.select("respondent"),on="respondent",how="left_anti"))

display("missing resp: ", missing_respondent.count())

StatementMeta(, df2ebdec-de60-466c-accb-07cbbccba455, 22, Finished, Available, Finished, False)

missing resp:  0


In [21]:
missing_date=(gold_fact_hour.select("date").distinct().join(dim_date.select("date"),on="date",how="left_anti"))
display("missing date:", missing_date.count())

StatementMeta(, df2ebdec-de60-466c-accb-07cbbccba455, 23, Finished, Available, Finished, False)

'missing date:'

In [44]:
fact_df = spark.table("Gold.gold_fact_hourly")
display(fact_df.limit(10))

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 46, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5028287c-b3c0-43e9-80d1-4a5678cdb27f)

In [47]:
spark.sql("DROP TABLE IF EXISTS Gold.gold_fact_hourl")

StatementMeta(, 079a1ffe-ea3a-4705-b6c9-c18e8914b79b, 49, Finished, Available, Finished, False)

DataFrame[]